# Data preparation

Normalize the supplied files, remove duplicate and conflicting text, reserve the supplied validation file as the final test set, and create reproducible group-disjoint training and validation splits.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import prepare_dataset_splits

In [2]:
columns = ["tweet_id", "entity", "sentiment", "text"]
training_df = pd.read_csv(
    PROJECT_ROOT / "data/twitter_training.csv",
    header=None,
    names=columns,
)
supplied_test_df = pd.read_csv(
    PROJECT_ROOT / "data/twitter_validation.csv",
    header=None,
    names=columns,
)

print(f"Supplied training rows: {len(training_df):,}")
print(f"Supplied test rows: {len(supplied_test_df):,}")

Supplied training rows: 74,682
Supplied test rows: 1,000


In [3]:
train_clean, validation_clean, test_clean, split_summary = (
    prepare_dataset_splits(training_df, supplied_test_df)
)

pd.Series(split_summary, name="rows")

training_rows_after_basic_cleaning     71308
duplicate_training_rows_removed         2408
conflicting_training_rows_removed        379
training_group_overlap_rows_removed     5533
training_text_overlap_rows_removed         5
test_rows_after_basic_cleaning          1000
duplicate_test_rows_removed                2
conflicting_test_rows_removed              0
train_rows                             50301
validation_rows                        12682
test_rows                                998
train_groups                            9018
validation_groups                       2255
test_groups                              998
Name: rows, dtype: int64

In [4]:
group_columns = ["entity", "tweet_id"]
train_groups = set(map(tuple, train_clean[group_columns].to_numpy()))
validation_groups = set(map(tuple, validation_clean[group_columns].to_numpy()))
test_groups = set(map(tuple, test_clean[group_columns].to_numpy()))

assert train_groups.isdisjoint(validation_groups)
assert train_groups.isdisjoint(test_groups)
assert validation_groups.isdisjoint(test_groups)
assert set(train_clean["text"]).isdisjoint(validation_clean["text"])
assert set(train_clean["text"]).isdisjoint(test_clean["text"])
assert set(validation_clean["text"]).isdisjoint(test_clean["text"])

split_distribution = pd.DataFrame({
    "train": train_clean["sentiment"].value_counts(normalize=True),
    "validation": validation_clean["sentiment"].value_counts(normalize=True),
    "test": test_clean["sentiment"].value_counts(normalize=True),
}).mul(100).round(2)

split_distribution

,train,validation,test
sentiment,,,
Irrelevant,17.59,17.57,17.23
Negative,30.88,30.74,26.55
Neutral,24.17,24.22,28.56
Positive,27.36,27.47,27.66


In [5]:
output_dir = PROJECT_ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_clean.to_csv(output_dir / "train_clean.csv", index=False)
validation_clean.to_csv(output_dir / "validation_clean.csv", index=False)
test_clean.to_csv(output_dir / "test_clean.csv", index=False)

print(f"Saved processed splits to {output_dir}")

Saved processed splits to C:\Users\jadka\OneDrive\Documents\progressSoft_internship\phase-1-machine-learning-nlp\assignment\data\processed


Each tweet ID has several closely related text variants. Keeping every entity and tweet-ID group in one split prevents those variants from inflating validation or test scores. Exact normalized text is also kept disjoint.